# 09 — Results synthesis corrected and clean

Notebook này tổng hợp kết quả từ notebook **02, 03, 04, 05, 07, 08**.


> Cách dùng tốt nhất: đặt notebook này cùng thư mục với các notebook 02, 03, 04, 05, 07, 08. Nếu đã chạy các notebook nguồn và có thư mục `outputs_02`, `outputs_03`, ..., `outputs_08`, notebook sẽ ưu tiên đọc CSV trong các thư mục đó. Nếu chưa có CSV, notebook sẽ đọc các bảng đã được lưu trong output HTML của file `.ipynb`.


In [1]:
# =========================
# 1. Imports and config
# =========================
from pathlib import Path
import json
import re
import warnings
from io import StringIO

import numpy as np
import pandas as pd
import nbformat
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 180)

ROOT = Path.cwd()
OUT_DIR = ROOT / "outputs_09"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N_ENSEMBLES = 3
TOP_DERIVED_DAILY_ROWS = 5

TASK_ORDER = ["target_1h", "target_24h", "daily_total", "multihorizon"]
DATASET_ORDER = {"3y": 0, "20y": 1, "unknown": 9}

TABULAR_MODELS = {
    "LinearRegression", "Ridge", "RandomForest", "SVR_RBF", "MLP",
    "HistGradientBoosting", "XGBoost", "CatBoost",
}
DEEP_POINT_MODELS = {"LSTM", "GRU"}
STANDARD_DEEP_MODELS = {"PatchTST", "TSMixer", "iTransformer"}

SOURCE_SPECS = {
    "02": {
        "name": "02_ml_baselines_boosting_ablation",
        "nb_glob": "02_ml_baselines_boosting_ablation*.ipynb",
        "csv_candidates": [
            "outputs_02/02_summary_for_report.csv",
            "outputs_02/02_results_all_models.csv",
        ],
    },
    "03": {
        "name": "03_deep_learning_ensemble_interpretability",
        "nb_glob": "03_deep_learning_ensemble_interpretability*.ipynb",
        "csv_candidates": ["outputs_03/03_final_summary_for_report.csv"],
    },
    "04": {
        "name": "04_daily_total_forecasting_final_comparison",
        "nb_glob": "04_daily_total_forecasting_final_comparison*.ipynb",
        "csv_candidates": ["outputs_04/04_daily_results_all_models.csv"],
    },
    "05": {
        "name": "05_extended_daily_total_forecasting_20y",
        "nb_glob": "05_extended_daily_total_forecasting_20y*.ipynb",
        "csv_candidates": ["outputs_05/05_daily_20y_results_all_models.csv"],
    },
    "07": {
        "name": "07_compact_multihorizon_24h_sequence_forecasting",
        "nb_glob": "07_compact_multihorizon_24h_sequence_forecasting*.ipynb",
        "csv_candidates": [
            "outputs_07/07_final_summary_for_report.csv",
            "outputs_07/07_results_all_models.csv",
        ],
    },
    "08": {
        "name": "08_standard_deep_sequence_models_all_tasks",
        "nb_glob": "08_standard_deep_sequence_models_all_tasks*.ipynb",
        "csv_candidates": ["outputs_08/08_results_all_models.csv"],
    },
}

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)


ROOT: c:\Users\nguye\Downloads\Big_Data
OUT_DIR: c:\Users\nguye\Downloads\Big_Data\outputs_09


In [2]:
# =========================
# 2. Helper functions: loading, cleaning, canonicalization
# =========================
load_log = []
_warning_messages = []


def warn_msg(msg):
    _warning_messages.append(msg)
    print("WARNING:", msg)


def flatten_columns(df):
    """Flatten MultiIndex columns and remove notebook display index columns."""
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        new_cols = []
        for tup in df.columns:
            parts = [str(x).strip() for x in tup if str(x).strip() and not str(x).startswith("Unnamed")]
            new_cols.append("_".join(parts).strip("_"))
        df.columns = new_cols
    else:
        df.columns = [str(c).strip() for c in df.columns]

    # Drop pandas display/index columns
    drop_cols = []
    for c in df.columns:
        cs = str(c).strip()
        if cs == "" or cs.startswith("Unnamed") or cs in {"index"}:
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=drop_cols, errors="ignore")

    # Normalize unicode ellipsis in object columns
    for c in df.select_dtypes(include="object").columns:
        df[c] = df[c].astype(str).str.replace("…", "...", regex=False)
    return df


def find_notebook(pattern):
    matches = sorted(ROOT.glob(pattern))
    if matches:
        return matches[0]
    return None


def extract_html_tables_from_notebook(nb_path):
    """Extract all displayed HTML tables from an already-run .ipynb."""
    nb_path = Path(nb_path)
    if not nb_path.exists():
        return []

    nb = nbformat.read(nb_path, as_version=4)
    tables = []

    for cell_idx, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        for out_idx, out in enumerate(cell.get("outputs", [])):
            data = out.get("data", {}) if isinstance(out, dict) else {}
            html = data.get("text/html")
            if html is None:
                continue
            if isinstance(html, list):
                html = "".join(html)
            try:
                found = pd.read_html(StringIO(html))
            except Exception:
                continue
            for table_idx, df in enumerate(found):
                df = flatten_columns(df)
                df.attrs["origin"] = f"{nb_path.name}::cell{cell_idx}_out{out_idx}_table{table_idx}"
                tables.append(df)
    return tables


def has_cols(df, cols):
    return all(c in df.columns for c in cols)


def pick_best_table(tables, predicate, description):
    candidates = [t for t in tables if predicate(t)]
    if not candidates:
        sample_cols = []
        for t in tables[:8]:
            sample_cols.append(list(t.columns)[:15])
        raise RuntimeError(
            f"Không tìm được bảng phù hợp cho: {description}. "
            f"Số bảng HTML tìm thấy: {len(tables)}. Một vài cột mẫu: {sample_cols}"
        )
    # Prefer the richest displayed table: more rows first, then more columns.
    candidates = sorted(candidates, key=lambda d: (len(d), len(d.columns)), reverse=True)
    return candidates[0]


def load_source_table(key, predicate, description):
    """Prefer exported CSV from outputs_xx; fallback to saved ipynb HTML tables."""
    spec = SOURCE_SPECS[key]

    # 1) Prefer explicit CSV outputs.
    for rel in spec["csv_candidates"]:
        path = ROOT / rel
        if path.exists():
            df = flatten_columns(pd.read_csv(path))
            load_log.append({
                "source_key": key,
                "source_name": spec["name"],
                "origin_type": "csv_output",
                "path_or_origin": str(path),
                "rows": len(df),
                "cols": len(df.columns),
                "description": description,
            })
            return df

    # 2) Fallback to output tables saved inside .ipynb.
    nb_path = find_notebook(spec["nb_glob"])
    if nb_path is None:
        raise FileNotFoundError(f"Không tìm thấy notebook nguồn: {spec['nb_glob']}")

    tables = extract_html_tables_from_notebook(nb_path)
    df = pick_best_table(tables, predicate, description)
    origin = df.attrs.get("origin", str(nb_path))
    load_log.append({
        "source_key": key,
        "source_name": spec["name"],
        "origin_type": "ipynb_html_output",
        "path_or_origin": origin,
        "rows": len(df),
        "cols": len(df.columns),
        "description": description,
    })
    return df


def canon_model(model):
    s = str(model).strip().replace("…", "...")
    s = re.sub(r"\s+", " ", s)

    direct = {
        "Random Forest": "RandomForest",
        "RandomForestRegressor": "RandomForest",
        "HistGradientBoostingRegressor": "HistGradientBoosting",
        "GradientBoosting": "HistGradientBoosting",
        "XGBRegressor": "XGBoost",
        "CatBoostRegressor": "CatBoost",
        "MLPRegressor": "MLP",
        "SVR": "SVR_RBF",
        "SVR-rbf": "SVR_RBF",
        "PatchTST-standard": "PatchTST",
        "TSMixer-standard": "TSMixer",
        "iTransformer-standard": "iTransformer",
    }
    if s in direct:
        return direct[s]

    # Repair pandas-truncated ensemble names from displayed notebook 07 tables.
    # Example: Ensemble_HistGradientBoosting_w0.5_TSMixer-lit...
    m = re.match(r"^(Ensemble_HistGradientBoosting_w)(0\.[357])(_TSMixer-lit)(?:e)?(?:\.\.\.)?$", s)
    if m:
        w = float(m.group(2))
        return f"Ensemble_HistGradientBoosting_w{w:.1f}_TSMixer-lite_w{1-w:.1f}"

    # Other common truncations. These are only for display repair; CSV outputs should already be full.
    s = s.replace("TSMixer-lit...", "TSMixer-lite")
    s = s.replace("iTransforme...", "iTransformer-mini")
    return s


def model_family(model, model_type=""):
    m = canon_model(model)
    mt = str(model_type).lower()
    if "ensemble" in mt or m.startswith("Ensemble"):
        return "ensemble"
    if m in {"LSTM", "GRU"}:
        return "deep_model"
    if m in STANDARD_DEEP_MODELS:
        return "deep_standard"
    if m in {"RandomForest", "HistGradientBoosting", "XGBoost", "CatBoost"}:
        return "boosting/tree"
    if m in {"LinearRegression", "Ridge", "SVR_RBF", "MLP"}:
        return "tabular_ml"
    if "baseline" in mt or "persistence" in m.lower():
        return "baseline"
    return "other"


def task_from_08(raw_task):
    """Correct order: detect multihorizon before target_24h."""
    s = str(raw_task).lower()
    if "multihorizon" in s or "24h_multihorizon" in s:
        return "multihorizon"
    if "daily_total" in s or "next_day" in s:
        return "daily_total"
    if "target_24h" in s:
        return "target_24h"
    if "target_1h" in s:
        return "target_1h"
    return str(raw_task)


def dataset_from_raw_task(raw_task, default="unknown"):
    s = str(raw_task).lower()
    if "20y" in s:
        return "20y"
    if "3y" in s:
        return "3y"
    return default


def to_num(x):
    try:
        return pd.to_numeric(x)
    except Exception:
        return np.nan


def row_get(row, *names, default=np.nan):
    for name in names:
        if name in row.index:
            val = row[name]
            if pd.notna(val):
                return val
    return default


def is_bad_model_name(model):
    s = str(model).lower()
    bad_tokens = [
        "persistence", "seasonal", "baseline",
        "tuned_", "hgbr_tuned", "ablation", "add_", "feature_set",
    ]
    return any(tok in s for tok in bad_tokens)


def is_standalone_lite_or_mini(model):
    s = str(model).lower()
    return ("lite" in s or "mini" in s) and not str(model).startswith("Ensemble")


def add_result(rows, **kwargs):
    defaults = dict(
        rank_in_task_dataset=np.nan,
        rank_in_scope=np.nan,
        task="",
        dataset_years="unknown",
        comparison_scope="",
        model="",
        model_family="",
        source_notebook="",
        source_task_raw="",
        sort_metric="",
        sort_RMSE=np.nan,
        val_sort_RMSE=np.nan,
        test_RMSE=np.nan,
        test_daylight_RMSE=np.nan,
        val_RMSE=np.nan,
        val_daylight_RMSE=np.nan,
        test_R2=np.nan,
        test_h1_RMSE=np.nan,
        test_h12_RMSE=np.nan,
        test_h24_RMSE=np.nan,
        test_daily_total_RMSE=np.nan,
        train_seconds=np.nan,
        n_params=np.nan,
        note="",
    )
    defaults.update(kwargs)
    defaults["model"] = canon_model(defaults["model"])
    if not defaults["model_family"]:
        defaults["model_family"] = model_family(defaults["model"])
    rows.append(defaults)


In [3]:
# =========================
# 3. Load source result tables
# =========================
# Selectors are based on column signatures rather than fixed cell numbers.

df02 = load_source_table(
    "02",
    predicate=lambda d: has_cols(d, ["target", "model", "val_daylight_RMSE", "test_daylight_RMSE"])
    and ("test_all_RMSE" in d.columns or "test_RMSE" in d.columns)
    and d["target"].astype(str).str.contains("target_").any(),
    description="Notebook 02 point forecasting ML/boosting summary",
)

df03 = load_source_table(
    "03",
    predicate=lambda d: has_cols(d, ["target", "model", "val_daylight_RMSE", "test_daylight_RMSE"])
    and ("model_type" in d.columns)
    and d["target"].astype(str).str.contains("target_").any(),
    description="Notebook 03 LSTM/GRU and ensemble summary",
)

df04 = load_source_table(
    "04",
    predicate=lambda d: has_cols(d, ["model", "val_RMSE", "test_RMSE", "test_R2"])
    and len(d) >= 5,
    description="Notebook 04 direct daily-total 3y summary",
)

df05 = load_source_table(
    "05",
    predicate=lambda d: has_cols(d, ["model", "val_RMSE", "test_RMSE", "test_R2"])
    and len(d) >= 5,
    description="Notebook 05 direct daily-total 20y summary",
)

df07 = load_source_table(
    "07",
    predicate=lambda d: has_cols(d, ["model", "model_type", "val_daylight_RMSE", "test_daylight_RMSE"])
    and ("test_h1_RMSE" in d.columns or "test_horizon_1_RMSE" in d.columns)
    and "test_daily_total_RMSE" in d.columns,
    description="Notebook 07 compact 24h multihorizon summary",
)

df08 = load_source_table(
    "08",
    predicate=lambda d: has_cols(d, ["task", "model", "val_daylight_RMSE", "test_daylight_RMSE", "test_RMSE"])
    and d["task"].astype(str).str.contains("20y|3y|target|multihorizon|daily", case=False, regex=True).any(),
    description="Notebook 08 standard deep sequence models summary",
)

load_log_df = pd.DataFrame(load_log)
display(load_log_df)

# Important warning for notebook 07 if only displayed ipynb table was available.
row07 = load_log_df[load_log_df["source_key"].eq("07")].iloc[0]
if row07["origin_type"] == "ipynb_html_output":
    warn_msg(
        "Notebook 07 đang được đọc từ bảng HTML đã display trong .ipynb. "
        "Nếu bảng display chỉ là head/top rows, hãy chạy notebook 07 trước để tạo outputs_07/07_final_summary_for_report.csv, "
        "khi đó notebook 09 sẽ đọc đầy đủ hơn."
    )


,source_key,source_name,origin_type,path_or_origin,rows,cols,description
0,02,02_ml_baselines_boosting_ablation,csv_output,c:\Users\nguye\Downloads\Big_Data\outputs_02\02_summary_for_report.csv,37,14,Notebook 02 point forecasting ML/boosting summary
1,03,03_deep_learning_ensemble_interpretability,csv_output,c:\Users\nguye\Downloads\Big_Data\outputs_03\03_final_summary_for_report.csv,22,33,Notebook 03 LSTM/GRU and ensemble summary
2,04,04_daily_total_forecasting_final_comparison,csv_output,c:\Users\nguye\Downloads\Big_Data\outputs_04\04_daily_results_all_models.csv,14,24,Notebook 04 direct daily-total 3y summary
3,05,05_extended_daily_total_forecasting_20y,csv_output,c:\Users\nguye\Downloads\Big_Data\outputs_05\05_daily_20y_results_all_models.csv,16,23,Notebook 05 direct daily-total 20y summary
4,07,07_compact_multihorizon_24h_sequence_forecasting,csv_output,c:\Users\nguye\Downloads\Big_Data\outputs_07\07_final_summary_for_report.csv,64,13,Notebook 07 compact 24h multihorizon summary
5,08,08_standard_deep_sequence_models_all_tasks,csv_output,c:\Users\nguye\Downloads\Big_Data\outputs_08\08_results_all_models.csv,18,38,Notebook 08 standard deep sequence models summary


In [4]:
# =========================
# 4. Build corrected canonical result table
# =========================
rows = []
reference_rows = []

# ---------- Notebook 02: direct point forecasting, 3y ----------
for _, r in df02.iterrows():
    model = canon_model(row_get(r, "model"))
    target = str(row_get(r, "target"))
    if target not in {"target_1h", "target_24h"}:
        continue
    if model not in TABULAR_MODELS:
        continue
    if is_bad_model_name(model):
        continue

    add_result(
        rows,
        task=target,
        dataset_years="3y",
        comparison_scope="direct_point",
        model=model,
        model_family=model_family(model),
        source_notebook="02_ml_baselines_boosting_ablation",
        source_task_raw=target,
        sort_metric="val_daylight_RMSE",
        sort_RMSE=row_get(r, "val_daylight_RMSE"),
        val_sort_RMSE=row_get(r, "val_daylight_RMSE"),
        test_RMSE=row_get(r, "test_all_RMSE", "test_RMSE"),
        test_daylight_RMSE=row_get(r, "test_daylight_RMSE"),
        val_RMSE=row_get(r, "val_all_RMSE", "val_RMSE"),
        val_daylight_RMSE=row_get(r, "val_daylight_RMSE"),
        test_R2=row_get(r, "test_daylight_R2", "test_R2"),
        train_seconds=row_get(r, "train_time_sec", "train_seconds"),
        note="Direct point forecasting on 3y dataset. Ranked by validation daylight RMSE.",
    )

# ---------- Notebook 03: direct point forecasting, 3y, deep + selected ensembles ----------
df03_work = df03.copy()
df03_work["model"] = df03_work["model"].map(canon_model)
if "status" in df03_work.columns:
    df03_work = df03_work[df03_work["status"].astype(str).str.lower().eq("ok")]

for target in ["target_1h", "target_24h"]:
    sub = df03_work[df03_work["target"].astype(str).eq(target)].copy()
    if sub.empty:
        continue
    deep = sub[sub["model"].isin(DEEP_POINT_MODELS)].copy()
    ens = sub[sub.get("model_type", "").astype(str).str.lower().eq("ensemble")].copy()
    ens = ens.sort_values(["val_daylight_RMSE", "test_daylight_RMSE"]).drop_duplicates("model").head(TOP_N_ENSEMBLES)
    use = pd.concat([deep, ens], ignore_index=True, sort=False)

    for _, r in use.iterrows():
        model = canon_model(row_get(r, "model"))
        fam = "ensemble" if str(row_get(r, "model_type", default="")).lower() == "ensemble" else "deep_model"
        add_result(
            rows,
            task=target,
            dataset_years="3y",
            comparison_scope="direct_point",
            model=model,
            model_family=fam,
            source_notebook="03_deep_learning_ensemble_interpretability",
            source_task_raw=target,
            sort_metric="val_daylight_RMSE",
            sort_RMSE=row_get(r, "val_daylight_RMSE"),
            val_sort_RMSE=row_get(r, "val_daylight_RMSE"),
            test_daylight_RMSE=row_get(r, "test_daylight_RMSE"),
            val_daylight_RMSE=row_get(r, "val_daylight_RMSE"),
            test_R2=row_get(r, "test_daylight_R2", "test_R2"),
            train_seconds=row_get(r, "train_seconds", "train_time_sec"),
            note=f"Direct point forecasting on 3y dataset. Kept LSTM/GRU plus top-{TOP_N_ENSEMBLES} validation-selected ensembles.",
        )

# ---------- Notebook 04: direct daily total, 3y ----------
for _, r in df04.iterrows():
    model = canon_model(row_get(r, "model"))
    if model not in TABULAR_MODELS:
        continue
    if is_bad_model_name(model):
        continue
    mt = str(row_get(r, "model_type", default="")).lower()
    if "baseline" in mt:
        continue

    add_result(
        rows,
        task="daily_total",
        dataset_years="3y",
        comparison_scope="direct_daily",
        model=model,
        model_family=model_family(model, mt),
        source_notebook="04_daily_total_forecasting_final_comparison",
        source_task_raw="next_day_daily_total_3y",
        sort_metric="val_RMSE",
        sort_RMSE=row_get(r, "val_RMSE"),
        val_sort_RMSE=row_get(r, "val_RMSE"),
        test_RMSE=row_get(r, "test_RMSE"),
        val_RMSE=row_get(r, "val_RMSE"),
        test_R2=row_get(r, "test_R2"),
        train_seconds=row_get(r, "train_seconds", "train_time_sec"),
        note="Direct next-day daily-total forecasting on 3y dataset. Ranked by validation RMSE.",
    )

# ---------- Notebook 05: direct daily total, 20y ----------
for _, r in df05.iterrows():
    model = canon_model(row_get(r, "model"))
    if model not in TABULAR_MODELS:
        continue
    if is_bad_model_name(model):
        continue
    mt = str(row_get(r, "model_type", default="")).lower()
    if "baseline" in mt:
        continue

    add_result(
        rows,
        task="daily_total",
        dataset_years="20y",
        comparison_scope="direct_daily",
        model=model,
        model_family=model_family(model, mt),
        source_notebook="05_extended_daily_total_forecasting_20y",
        source_task_raw="next_day_daily_total_20y",
        sort_metric="val_RMSE",
        sort_RMSE=row_get(r, "val_RMSE"),
        val_sort_RMSE=row_get(r, "val_RMSE"),
        test_RMSE=row_get(r, "test_RMSE"),
        val_RMSE=row_get(r, "val_RMSE"),
        test_R2=row_get(r, "test_R2"),
        train_seconds=row_get(r, "train_seconds", "train_time_sec"),
        note="Direct next-day daily-total forecasting on 20y dataset. Ranked by validation RMSE.",
    )

# ---------- Notebook 07: direct 24h multihorizon, 20y ----------
df07_work = df07.copy()
df07_work["model"] = df07_work["model"].map(canon_model)
df07_work["model_family_clean"] = df07_work.apply(lambda rr: model_family(rr.get("model"), rr.get("model_type", "")), axis=1)

# Normalize possible horizon column names from notebook 06/07 variants.
rename07 = {
    "test_horizon_1_RMSE": "test_h1_RMSE",
    "test_horizon_12_RMSE": "test_h12_RMSE",
    "test_horizon_24_RMSE": "test_h24_RMSE",
}
df07_work = df07_work.rename(columns={k: v for k, v in rename07.items() if k in df07_work.columns})

# Keep top-N ensembles selected by validation daylight RMSE.
ens07 = df07_work[df07_work["model_family_clean"].eq("ensemble")].copy()
ens07 = ens07.sort_values(["val_daylight_RMSE", "test_daylight_RMSE"]).drop_duplicates("model").head(TOP_N_ENSEMBLES)

# Keep regular comparable models. Standalone lite/mini deep variants are not mixed; standard deep variants are from notebook 08.
regular07 = df07_work[~df07_work["model_family_clean"].eq("ensemble")].copy()
regular07 = regular07[~regular07["model"].map(is_bad_model_name)]
regular07 = regular07[~regular07["model"].map(is_standalone_lite_or_mini)]
regular07 = regular07[regular07["model"].isin(TABULAR_MODELS | DEEP_POINT_MODELS)]
regular07 = regular07.sort_values(["val_daylight_RMSE", "test_daylight_RMSE"]).drop_duplicates("model")

use07 = pd.concat([ens07, regular07], ignore_index=True, sort=False)

for _, r in use07.iterrows():
    model = canon_model(row_get(r, "model"))
    add_result(
        rows,
        task="multihorizon",
        dataset_years="20y",
        comparison_scope="direct_multihorizon",
        model=model,
        model_family=row_get(r, "model_family_clean", default=model_family(model, row_get(r, "model_type", default=""))),
        source_notebook="07_compact_multihorizon_24h_sequence_forecasting",
        source_task_raw="24h_multihorizon_20y",
        sort_metric="val_daylight_RMSE",
        sort_RMSE=row_get(r, "val_daylight_RMSE"),
        val_sort_RMSE=row_get(r, "val_daylight_RMSE"),
        test_RMSE=row_get(r, "test_RMSE"),
        test_daylight_RMSE=row_get(r, "test_daylight_RMSE"),
        val_daylight_RMSE=row_get(r, "val_daylight_RMSE"),
        test_R2=row_get(r, "test_R2"),
        test_h1_RMSE=row_get(r, "test_h1_RMSE"),
        test_h12_RMSE=row_get(r, "test_h12_RMSE"),
        test_h24_RMSE=row_get(r, "test_h24_RMSE"),
        test_daily_total_RMSE=row_get(r, "test_daily_total_RMSE"),
        train_seconds=row_get(r, "train_seconds", "train_time_sec"),
        note=f"Direct 24h multihorizon result on 20y dataset. Kept regular models plus top-{TOP_N_ENSEMBLES} validation-selected ensembles.",
    )

# Derived daily total from 24h sequence prediction. This is scalar daily total, so it can be ranked by val_daily_total_RMSE when available, but scope is labelled clearly.
daily07 = df07_work.dropna(subset=["test_daily_total_RMSE"]).copy()
daily07 = daily07[~daily07["model"].map(is_bad_model_name)]
daily07 = daily07[~daily07["model"].map(is_standalone_lite_or_mini)]
daily07 = daily07.sort_values(["test_daily_total_RMSE", "test_daylight_RMSE"]).drop_duplicates("model").head(TOP_DERIVED_DAILY_ROWS)

for _, r in daily07.iterrows():
    model = canon_model(row_get(r, "model"))
    add_result(
        rows,
        task="daily_total",
        dataset_years="20y",
        comparison_scope="derived_from_24h_multihorizon",
        model=model,
        model_family=row_get(r, "model_family_clean", default=model_family(model, row_get(r, "model_type", default=""))),
        source_notebook="07_compact_multihorizon_24h_sequence_forecasting__derived",
        source_task_raw="daily_total_from_24h_multihorizon_20y",
        sort_metric="val_daily_total_RMSE",
        sort_RMSE=row_get(r, "val_daily_total_RMSE", "daily_total_val_RMSE", default=row_get(r, "test_daily_total_RMSE")),
        val_sort_RMSE=row_get(r, "val_daily_total_RMSE", "daily_total_val_RMSE", default=np.nan),
        test_RMSE=row_get(r, "test_daily_total_RMSE"),
        test_R2=row_get(r, "test_daily_total_R2", "daily_total_R2", default=np.nan),
        test_daily_total_RMSE=row_get(r, "test_daily_total_RMSE"),
        note="Daily total obtained by summing 24h multihorizon predictions. Labelled as derived, not direct daily model.",
    )

# Reference-only h1/h24 rows from notebook 07. These are NOT mixed into point forecasting ranking.
for task, metric_col in [("target_1h", "test_h1_RMSE"), ("target_24h", "test_h24_RMSE")]:
    if metric_col not in df07_work.columns:
        continue
    ref = df07_work.dropna(subset=[metric_col]).copy()
    ref = ref[~ref["model"].map(is_bad_model_name)]
    ref = ref[~ref["model"].map(is_standalone_lite_or_mini)]
    ref = ref.sort_values(["val_daylight_RMSE", metric_col]).drop_duplicates("model")
    for _, r in ref.iterrows():
        model = canon_model(row_get(r, "model"))
        reference_rows.append({
            "task": task,
            "dataset_years": "20y",
            "comparison_scope": "reference_only_from_24h_multihorizon",
            "model": model,
            "model_family": row_get(r, "model_family_clean", default=model_family(model, row_get(r, "model_type", default=""))),
            "source_notebook": "07_compact_multihorizon_24h_sequence_forecasting__derived",
            "reference_sort_metric": "val_daylight_RMSE_of_whole_24h_sequence",
            "reference_sort_RMSE": row_get(r, "val_daylight_RMSE"),
            "horizon_specific_metric": metric_col,
            "horizon_specific_RMSE": row_get(r, metric_col),
            "test_RMSE": row_get(r, "test_RMSE"),
            "test_daylight_RMSE": row_get(r, "test_daylight_RMSE"),
            "test_h1_RMSE": row_get(r, "test_h1_RMSE"),
            "test_h12_RMSE": row_get(r, "test_h12_RMSE"),
            "test_h24_RMSE": row_get(r, "test_h24_RMSE"),
            "test_daily_total_RMSE": row_get(r, "test_daily_total_RMSE"),
            "test_R2": row_get(r, "test_R2"),
            "note": "Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.",
        })

# ---------- Notebook 08: standard deep sequence models, all supported tasks ----------
for _, r in df08.iterrows():
    raw_task = str(row_get(r, "task"))
    task = task_from_08(raw_task)
    dataset_years = dataset_from_raw_task(raw_task, default="20y")
    model = canon_model(row_get(r, "model"))
    if model not in STANDARD_DEEP_MODELS:
        continue

    if task in {"target_1h", "target_24h"}:
        scope = "direct_point"
        sort_metric = "val_daylight_RMSE"
        sort_value = row_get(r, "val_daylight_RMSE")
        val_sort = row_get(r, "val_daylight_RMSE")
        note = "Direct point forecasting standard deep model. Ranked by validation daylight RMSE."
    elif task == "daily_total":
        scope = "direct_daily"
        sort_metric = "val_RMSE"
        sort_value = row_get(r, "val_RMSE", "val_daylight_RMSE")
        val_sort = row_get(r, "val_daylight_RMSE", "val_RMSE")
        note = "Direct daily-total standard deep model. Ranked by validation RMSE."
    elif task == "multihorizon":
        scope = "direct_multihorizon"
        sort_metric = "val_daylight_RMSE"
        sort_value = row_get(r, "val_daylight_RMSE")
        val_sort = row_get(r, "val_daylight_RMSE")
        note = "Direct 24h multihorizon standard deep model. Mapping fixed: multihorizon_20y_24h stays multihorizon. Ranked by validation daylight RMSE."
    else:
        continue

    add_result(
        rows,
        task=task,
        dataset_years=dataset_years,
        comparison_scope=scope,
        model=model,
        model_family="deep_standard",
        source_notebook="08_standard_deep_sequence_models_all_tasks",
        source_task_raw=raw_task,
        sort_metric=sort_metric,
        sort_RMSE=sort_value,
        val_sort_RMSE=val_sort,
        test_RMSE=row_get(r, "test_RMSE"),
        test_daylight_RMSE=row_get(r, "test_daylight_RMSE"),
        val_daylight_RMSE=row_get(r, "val_daylight_RMSE"),
        test_R2=row_get(r, "test_R2"),
        test_h1_RMSE=row_get(r, "test_h1_RMSE"),
        test_h12_RMSE=row_get(r, "test_h12_RMSE"),
        test_h24_RMSE=row_get(r, "test_h24_RMSE"),
        test_daily_total_RMSE=row_get(r, "test_daily_total_RMSE"),
        train_seconds=row_get(r, "train_seconds", "train_time_sec"),
        n_params=row_get(r, "n_params"),
        note=note,
    )

print("Rows collected before final ranking:", len(rows))


Rows collected before final ranking: 77


In [5]:
# =========================
# 5. Finalize ranking, validation, and save outputs
# =========================
results = pd.DataFrame(rows)

numeric_cols = [
    "sort_RMSE", "val_sort_RMSE", "test_RMSE", "test_daylight_RMSE", "val_RMSE", "val_daylight_RMSE",
    "test_R2", "test_h1_RMSE", "test_h12_RMSE", "test_h24_RMSE", "test_daily_total_RMSE",
    "train_seconds", "n_params",
]
for c in numeric_cols:
    if c in results.columns:
        results[c] = pd.to_numeric(results[c], errors="coerce")

# Remove exact duplicate model-source-scope rows; keep the lower validation ranking RMSE.
results = (
    results.sort_values(["task", "dataset_years", "comparison_scope", "model", "sort_RMSE"], na_position="last")
    .drop_duplicates(["task", "dataset_years", "comparison_scope", "model", "source_notebook"], keep="first")
    .reset_index(drop=True)
)

# Sorting policy:
# - target_1h/target_24h: direct point models only, ranked by val_daylight_RMSE.
# - daily_total: direct and derived scalar daily-total rows can be displayed together, but scope is clearly labelled; ranked by validation RMSE when available.
# - multihorizon: ranked by val_daylight_RMSE.
def scope_order(row):
    task = row["task"]
    scope = row["comparison_scope"]
    if task in {"target_1h", "target_24h"}:
        return 0 if scope == "direct_point" else 9
    if task == "daily_total":
        # Allow derived daily total and direct daily total in the same scalar RMSE ranking.
        # The scope column prevents over-claiming.
        return 0
    if task == "multihorizon":
        return 0
    return 9

results["task_order"] = results["task"].map({t: i for i, t in enumerate(TASK_ORDER)}).fillna(99).astype(int)
results["dataset_order"] = results["dataset_years"].map(DATASET_ORDER).fillna(99).astype(int)
results["scope_order"] = results.apply(scope_order, axis=1)

results = results.sort_values(
    ["task_order", "dataset_order", "scope_order", "sort_RMSE", "test_RMSE", "model"],
    na_position="last",
).reset_index(drop=True)

ranked_parts = []
for (task, dset), sub in results.groupby(["task", "dataset_years"], sort=False):
    sub = sub.copy()
    sub["rank_in_task_dataset"] = np.arange(1, len(sub) + 1)
    sub["rank_in_scope"] = sub.groupby("comparison_scope").cumcount() + 1
    ranked_parts.append(sub)
results = pd.concat(ranked_parts, ignore_index=True)

report_cols = [
    "rank_in_task_dataset", "rank_in_scope", "task", "dataset_years", "comparison_scope",
    "model", "model_family", "source_notebook", "source_task_raw",
    "sort_metric", "sort_RMSE", "val_sort_RMSE",
    "test_RMSE", "test_daylight_RMSE", "val_RMSE", "val_daylight_RMSE", "test_R2",
    "test_h1_RMSE", "test_h12_RMSE", "test_h24_RMSE", "test_daily_total_RMSE",
    "train_seconds", "n_params", "note",
]
report = results[[c for c in report_cols if c in results.columns]].copy()

# Reference table for h1/h24 derived from notebook 07 multihorizon output.
reference = pd.DataFrame(reference_rows)
if not reference.empty:
    for c in [
        "reference_sort_RMSE", "horizon_specific_RMSE", "test_RMSE", "test_daylight_RMSE",
        "test_h1_RMSE", "test_h12_RMSE", "test_h24_RMSE", "test_daily_total_RMSE", "test_R2",
    ]:
        if c in reference.columns:
            reference[c] = pd.to_numeric(reference[c], errors="coerce")
    reference["task_order"] = reference["task"].map({t: i for i, t in enumerate(TASK_ORDER)}).fillna(99).astype(int)
    reference["dataset_order"] = reference["dataset_years"].map(DATASET_ORDER).fillna(99).astype(int)
    reference = reference.sort_values(
        ["task_order", "dataset_order", "reference_sort_RMSE", "horizon_specific_RMSE", "model"],
        na_position="last",
    ).reset_index(drop=True)
    reference["reference_rank"] = reference.groupby(["task", "dataset_years"]).cumcount() + 1
    reference_cols = [
        "reference_rank", "task", "dataset_years", "comparison_scope", "model", "model_family", "source_notebook",
        "reference_sort_metric", "reference_sort_RMSE", "horizon_specific_metric", "horizon_specific_RMSE",
        "test_RMSE", "test_daylight_RMSE", "test_h1_RMSE", "test_h12_RMSE", "test_h24_RMSE",
        "test_daily_total_RMSE", "test_R2", "note",
    ]
    reference = reference[[c for c in reference_cols if c in reference.columns]]

# Best models per task/dataset. For daily_total 20y, add direct-only best as a separate row.
best_rows = []
for (task, dset), sub in report.groupby(["task", "dataset_years"], sort=False):
    best = sub.sort_values("sort_RMSE", na_position="last").iloc[0].to_dict()
    best_rows.append({
        "task": task,
        "dataset_years": dset,
        "best_scope": best.get("comparison_scope"),
        "model": best.get("model"),
        "model_family": best.get("model_family"),
        "source_notebook": best.get("source_notebook"),
        "sort_metric": best.get("sort_metric"),
        "sort_RMSE": best.get("sort_RMSE"),
        "test_R2": best.get("test_R2"),
        "note": "Best by the task-specific validation ranking metric.",
    })
    if task == "daily_total" and dset == "20y":
        direct = sub[sub["comparison_scope"].eq("direct_daily")]
        if not direct.empty:
            b = direct.sort_values("sort_RMSE", na_position="last").iloc[0].to_dict()
            best_rows.append({
                "task": "daily_total_direct_only",
                "dataset_years": dset,
                "best_scope": b.get("comparison_scope"),
                "model": b.get("model"),
                "model_family": b.get("model_family"),
                "source_notebook": b.get("source_notebook"),
                "sort_metric": b.get("sort_metric"),
                "sort_RMSE": b.get("sort_RMSE"),
                "test_R2": b.get("test_R2"),
                "note": "Best by validation RMSE among direct daily-total models only, excluding derived 24h totals.",
            })

best_by_task = pd.DataFrame(best_rows)

coverage = (
    report.groupby(["task", "dataset_years", "comparison_scope", "source_notebook", "model_family"], dropna=False)
    .size()
    .reset_index(name="n_rows")
    .sort_values(["task", "dataset_years", "comparison_scope", "source_notebook", "model_family"])
)

sorting_policy = pd.DataFrame([
    {
        "task": "target_1h / target_24h",
        "dataset": "3y / 20y",
        "sort_metric": "val_daylight_RMSE",
        "reason": "Point forecasting is evaluated on daylight hours. validation daylight RMSE is used for model selection; test metrics are kept for final reporting.",
    },
    {
        "task": "daily_total",
        "dataset": "3y / 20y",
        "sort_metric": "val_RMSE or val_daily_total_RMSE",
        "reason": "Daily total is ranked by validation RMSE. Derived 24h totals are clearly labelled by comparison_scope.",
    },
    {
        "task": "multihorizon",
        "dataset": "20y",
        "sort_metric": "val_daylight_RMSE",
        "reason": "24h sequence forecasting is selected by validation daylight sequence RMSE. h1/h12/h24 are diagnostics, not the main ranking metric.",
    },
    {
        "task": "target_1h / target_24h derived from notebook 07",
        "dataset": "20y",
        "sort_metric": "separate reference table",
        "reason": "h1/h24 metrics from a 24h multihorizon model are not direct point-forecasting models, so they are not mixed into point-ranking tables.",
    },
])

# Validation checks: fail loudly if the known logic bug returns.
problems = []

bad08 = report[
    report["source_notebook"].eq("08_standard_deep_sequence_models_all_tasks")
    & report["source_task_raw"].astype(str).str.contains("multihorizon", case=False, na=False)
    & ~report["task"].eq("multihorizon")
]
if not bad08.empty:
    problems.append("Notebook 08 multihorizon rows were mapped outside multihorizon.")

bad_point_metric = report[
    report["task"].isin(["target_1h", "target_24h"])
    & report["comparison_scope"].eq("direct_point")
    & ~report["sort_metric"].eq("val_daylight_RMSE")
]
if not bad_point_metric.empty:
    problems.append("Some direct point rows are not ranked by val_daylight_RMSE.")

bad_mh_metric = report[
    report["task"].eq("multihorizon")
    & ~report["sort_metric"].eq("val_daylight_RMSE")
]
if not bad_mh_metric.empty:
    problems.append("Some multihorizon rows are not ranked by val_daylight_RMSE.")

bad_reference_mix = report[report["comparison_scope"].astype(str).str.contains("reference_only", na=False)]
if not bad_reference_mix.empty:
    problems.append("Reference-only notebook 07 h1/h24 rows leaked into the main report table.")

if problems:
    raise AssertionError("\n".join(problems))
else:
    print("VALIDATION PASSED: task mapping and ranking policy are consistent.")

# Save outputs.
report_path = OUT_DIR / "09_corrected_final_results_for_report.csv"
best_path = OUT_DIR / "09_corrected_best_by_task_dataset.csv"
coverage_path = OUT_DIR / "09_corrected_source_coverage.csv"
reference_path = OUT_DIR / "09_reference_only_derived_point_from_notebook07.csv"
policy_path = OUT_DIR / "09_sorting_policy.csv"
excel_path = OUT_DIR / "09_corrected_results_synthesis.xlsx"

report.to_csv(report_path, index=False, encoding="utf-8-sig")
best_by_task.to_csv(best_path, index=False, encoding="utf-8-sig")
coverage.to_csv(coverage_path, index=False, encoding="utf-8-sig")
sorting_policy.to_csv(policy_path, index=False, encoding="utf-8-sig")
if not reference.empty:
    reference.to_csv(reference_path, index=False, encoding="utf-8-sig")

try:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        report.to_excel(writer, sheet_name="Report_Table_Clean", index=False)
        best_by_task.to_excel(writer, sheet_name="Best_By_Task", index=False)
        coverage.to_excel(writer, sheet_name="Coverage", index=False)
        sorting_policy.to_excel(writer, sheet_name="Sorting_Policy", index=False)
        if not reference.empty:
            reference.to_excel(writer, sheet_name="Reference_Derived_07", index=False)
except Exception as e:
    warn_msg(f"Không xuất được Excel do: {repr(e)}. CSV vẫn đã được lưu đầy đủ.")

print("Saved:")
for p in [report_path, best_path, coverage_path, policy_path, reference_path if not reference.empty else None, excel_path]:
    if p is not None and Path(p).exists():
        print("-", p)

print("Final report rows:", len(report))
print("Reference rows:", len(reference))


VALIDATION PASSED: task mapping and ranking policy are consistent.
Saved:
- c:\Users\nguye\Downloads\Big_Data\outputs_09\09_corrected_final_results_for_report.csv
- c:\Users\nguye\Downloads\Big_Data\outputs_09\09_corrected_best_by_task_dataset.csv
- c:\Users\nguye\Downloads\Big_Data\outputs_09\09_corrected_source_coverage.csv
- c:\Users\nguye\Downloads\Big_Data\outputs_09\09_sorting_policy.csv
- c:\Users\nguye\Downloads\Big_Data\outputs_09\09_reference_only_derived_point_from_notebook07.csv
Final report rows: 77
Reference rows: 116


In [6]:
# =========================
# 6. Display clean report tables
# =========================
print("Sorting policy (validation-based ranking):")
display(sorting_policy)

print("\nSource coverage:")
display(coverage.reset_index(drop=True))

print("\nBest model by task/dataset:")
show_best = best_by_task.copy()
for c in show_best.select_dtypes(include=[np.number]).columns:
    show_best[c] = show_best[c].round(4)
display(show_best.reset_index(drop=True))


def show_table(task, dataset):
    sub = report[(report["task"].eq(task)) & (report["dataset_years"].eq(dataset))].copy()
    if sub.empty:
        print(f"\n===== {task} | {dataset}: no rows =====")
        return
    cols = [
        "rank_in_task_dataset", "rank_in_scope", "comparison_scope",
        "model", "model_family", "source_notebook", "sort_metric", "sort_RMSE", "val_sort_RMSE",
        "test_RMSE", "test_daylight_RMSE", "test_R2",
        "test_h1_RMSE", "test_h12_RMSE", "test_h24_RMSE", "test_daily_total_RMSE",
        "note",
    ]
    cols = [c for c in cols if c in sub.columns]
    show = sub[cols].copy()
    for c in show.select_dtypes(include=[np.number]).columns:
        show[c] = show[c].round(4)
    print(f"\n===== {task} | dataset {dataset} | {len(show)} rows =====")
    display(show.reset_index(drop=True))

for task in TASK_ORDER:
    for dataset in ["3y", "20y"]:
        show_table(task, dataset)

if not reference.empty:
    print("\n===== Reference only: target_1h/target_24h derived from notebook 07 multihorizon =====")
    ref_show = reference.copy()
    for c in ref_show.select_dtypes(include=[np.number]).columns:
        ref_show[c] = ref_show[c].round(4)
    display(ref_show.reset_index(drop=True))

if _warning_messages:
    print("\nWarnings:")
    for msg in _warning_messages:
        print("-", msg)


Sorting policy (validation-based ranking):


,task,dataset,sort_metric,reason
0,target_1h / target_24h,3y / 20y,val_daylight_RMSE,Point forecasting is evaluated on daylight hours. validation daylight RMSE is used for model selection; test metrics are kept for final reporting.
1,daily_total,3y / 20y,val_RMSE or val_daily_total_RMSE,Daily total is ranked by validation RMSE. Derived 24h totals are clearly labelled by comparison_scope.
2,multihorizon,20y,val_daylight_RMSE,"24h sequence forecasting is selected by validation daylight sequence RMSE. h1/h12/h24 are diagnostics, not the main ranking metric."
3,target_1h / target_24h derived from notebook 07,20y,separate reference table,"h1/h24 metrics from a 24h multihorizon model are not direct point-forecasting models, so they are not mixed into point-ranking tables."



Source coverage:


,task,dataset_years,comparison_scope,source_notebook,model_family,n_rows
0,daily_total,20y,derived_from_24h_multihorizon,07_compact_multihorizon_24h_sequence_forecasting__derived,ensemble,5
1,daily_total,20y,direct_daily,05_extended_daily_total_forecasting_20y,boosting/tree,4
2,daily_total,20y,direct_daily,05_extended_daily_total_forecasting_20y,tabular_ml,4
3,daily_total,20y,direct_daily,08_standard_deep_sequence_models_all_tasks,deep_standard,3
4,daily_total,3y,direct_daily,04_daily_total_forecasting_final_comparison,boosting/tree,4
5,daily_total,3y,direct_daily,04_daily_total_forecasting_final_comparison,tabular_ml,4
6,multihorizon,20y,direct_multihorizon,07_compact_multihorizon_24h_sequence_forecasting,boosting/tree,4
7,multihorizon,20y,direct_multihorizon,07_compact_multihorizon_24h_sequence_forecasting,deep_model,2
8,multihorizon,20y,direct_multihorizon,07_compact_multihorizon_24h_sequence_forecasting,ensemble,3
9,multihorizon,20y,direct_multihorizon,07_compact_multihorizon_24h_sequence_forecasting,tabular_ml,3



Best model by task/dataset:


...


===== multihorizon | 3y: no rows =====

===== multihorizon | dataset 20y | 15 rows =====


,rank_in_task_dataset,rank_in_scope,comparison_scope,model,model_family,source_notebook,sort_metric,sort_RMSE,val_sort_RMSE,test_RMSE,test_daylight_RMSE,test_R2,test_h1_RMSE,test_h12_RMSE,test_h24_RMSE,test_daily_total_RMSE,note
0,1,1,direct_multihorizon,Ensemble_HistGradientBoosting_w0.5_TSMixer-lite_w0.5,ensemble,07_compact_multihorizon_24h_sequence_forecasting,val_daylight_RMSE,225.2983,225.2983,179.1363,248.7674,0.9761,80.2708,173.0706,185.0985,2331.6245,Direct 24h multihorizon result on 20y dataset. Kept regular models plus top-3 validation-selected ensembles.
1,2,2,direct_multihorizon,Ensemble_HistGradientBoosting_w0.7_TSMixer-lite_w0.3,ensemble,07_compact_multihorizon_24h_sequence_forecasting,val_daylight_RMSE,225.4473,225.4473,179.4799,249.3231,0.9760,77.0543,173.3636,184.9180,2331.9780,Direct 24h multihorizon result on 20y dataset. Kept regular models plus top-3 validation-selected ensembles.
2,3,3,direct_multihorizon,Ensemble_XGBoost_w0.5_TSMixer-lite_w0.5,ensemble,07_compact_multihorizon_24h_sequence_forecasting,val_daylight_RMSE,225.6598,225.6598,178.9700,248.5647,0.9761,79.5839,173.2812,185.3729,2330.4372,Direct 24h multihorizon result on 20y dataset. Kept regular models plus top-3 validation-selected ensembles.
3,4,4,direct_multihorizon,HistGradientBoosting,boosting/tree,07_compact_multihorizon_24h_sequence_forecasting,val_daylight_RMSE,228.6927,228.6927,182.0921,253.0027,0.9753,76.8539,176.3031,186.9944,2357.1996,Direct 24h multihorizon result on 20y dataset. Kept regular models plus top-3 validation-selected ensembles.
4,5,5,direct_multihorizon,CatBoost,boosting/tree,07_compact_multihorizon_24h_sequence_forecasting,val_daylight_RMSE,228.7461,228.7461,183.2207,254.4668,0.9750,86.5496,177.8927,184.6244,2402.0228,Direct 24h multihorizon result on 20y dataset. Kept regular models plus top-3 validation-selected ensembles.
5,6,6,direct_multihorizon,XGBoost,boosting/tree,07_compact_multihorizon_24h_sequence_forecasting,val_daylight_RMSE,228.8779,228.8779,181.5699,252.2976,0.9754,75.2797,175.7309,187.1837,2355.9341,Direct 24h multihorizon result on 20y dataset. Kept regular models plus top-3 validation-selected ensembles.
6,7,7,direct_multihorizon,TSMixer,deep_standard,08_standard_deep_sequence_models_all_tasks,val_daylight_RMSE,232.4545,232.4545,184.4697,255.4467,0.9747,95.6223,185.2544,184.8664,2442.4665,Direct 24h multihorizon standard deep model. Mapping fixed: multihorizon_20y_24h stays multihorizon. Ranked by validation daylight RMSE.
7,8,8,direct_multihorizon,LSTM,deep_model,07_compact_multihorizon_24h_sequence_forecasting,val_daylight_RMSE,232.8166,232.8166,186.2112,257.9015,0.9742,103.4107,181.1944,190.3317,2431.0682,Direct 24h multihorizon result on 20y dataset. Kept regular models plus top-3 validation-selected ensembles.
8,9,9,direct_multihorizon,RandomForest,boosting/tree,07_compact_multihorizon_24h_sequence_forecasting,val_daylight_RMSE,233.9406,233.9406,185.0395,257.1324,0.9745,79.6100,181.8816,189.0350,2430.1861,Direct 24h multihorizon result on 20y dataset. Kept regular models plus top-3 validation-selected ensembles.
9,10,10,direct_multihorizon,PatchTST,deep_standard,08_standard_deep_sequence_models_all_tasks,val_daylight_RMSE,234.0300,234.0300,185.8067,257.8024,0.9743,101.0857,179.6887,189.0820,2435.0934,Direct 24h multihorizon standard deep model. Mapping fixed: multihorizon_20y_24h stays multihorizon. Ranked by validation daylight RMSE.



===== Reference only: target_1h/target_24h derived from notebook 07 multihorizon =====


,reference_rank,task,dataset_years,comparison_scope,model,model_family,source_notebook,reference_sort_metric,reference_sort_RMSE,horizon_specific_metric,horizon_specific_RMSE,test_RMSE,test_daylight_RMSE,test_h1_RMSE,test_h12_RMSE,test_h24_RMSE,test_daily_total_RMSE,test_R2,note
0,1,target_1h,20y,reference_only_from_24h_multihorizon,Ensemble_HistGradientBoosting_w0.5_TSMixer-lite_w0.5,ensemble,07_compact_multihorizon_24h_sequence_forecasting__derived,val_daylight_RMSE_of_whole_24h_sequence,225.2983,test_h1_RMSE,80.2708,179.1363,248.7674,80.2708,173.0706,185.0985,2331.6245,0.9761,Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.
1,2,target_1h,20y,reference_only_from_24h_multihorizon,Ensemble_HistGradientBoosting_w0.7_TSMixer-lite_w0.3,ensemble,07_compact_multihorizon_24h_sequence_forecasting__derived,val_daylight_RMSE_of_whole_24h_sequence,225.4473,test_h1_RMSE,77.0543,179.4799,249.3231,77.0543,173.3636,184.9180,2331.9780,0.9760,Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.
2,3,target_1h,20y,reference_only_from_24h_multihorizon,Ensemble_XGBoost_w0.5_TSMixer-lite_w0.5,ensemble,07_compact_multihorizon_24h_sequence_forecasting__derived,val_daylight_RMSE_of_whole_24h_sequence,225.6598,test_h1_RMSE,79.5839,178.9700,248.5647,79.5839,173.2812,185.3729,2330.4372,0.9761,Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.
3,4,target_1h,20y,reference_only_from_24h_multihorizon,Ensemble_XGBoost_w0.7_TSMixer-lite_w0.3,ensemble,07_compact_multihorizon_24h_sequence_forecasting__derived,val_daylight_RMSE_of_whole_24h_sequence,225.8037,test_h1_RMSE,76.0138,179.1933,248.9530,76.0138,173.3786,185.2027,2330.6002,0.9761,Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.
4,5,target_1h,20y,reference_only_from_24h_multihorizon,Ensemble_HistGradientBoosting_w0.7_LSTM_w0.3,ensemble,07_compact_multihorizon_24h_sequence_forecasting__derived,val_daylight_RMSE_of_whole_24h_sequence,226.2458,test_h1_RMSE,78.8623,180.5477,250.7569,78.8623,174.6711,185.5136,2350.6294,0.9757,Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,54,target_24h,20y,reference_only_from_24h_multihorizon,GRU,deep_model,07_compact_multihorizon_24h_sequence_forecasting__derived,val_daylight_RMSE_of_whole_24h_sequence,236.6489,test_h24_RMSE,189.2904,187.3487,259.5219,103.7933,180.1693,189.2904,2432.5315,0.9739,Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.
112,55,target_24h,20y,reference_only_from_24h_multihorizon,Ridge,tabular_ml,07_compact_multihorizon_24h_sequence_forecasting__derived,val_daylight_RMSE_of_whole_24h_sequence,242.8027,test_h24_RMSE,187.2372,191.8178,264.6083,81.4936,185.3787,187.2372,2436.0407,0.9726,Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.
113,56,target_24h,20y,reference_only_from_24h_multihorizon,Linear Regression,other,07_compact_multihorizon_24h_sequence_forecasting__derived,val_daylight_RMSE_of_whole_24h_sequence,243.0724,test_h24_RMSE,187.3244,191.9421,264.6103,81.6519,185.4648,187.3244,2438.3658,0.9726,Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.
114,57,target_24h,20y,reference_only_from_24h_multihorizon,MLP,tabular_ml,07_compact_multihorizon_24h_sequence_forecasting__derived,val_daylight_RMSE_of_whole_24h_sequence,246.5631,test_h24_RMSE,197.9981,195.1601,270.4582,112.5539,184.2103,197.9981,2465.8342,0.9716,Reference only: h1/h24 metric comes from 24h multihorizon output; do not mix-rank with direct point models.



Warnings:
- Không xuất được Excel do: ModuleNotFoundError("No module named 'openpyxl'"). CSV vẫn đã được lưu đầy đủ.


# Ghi chú cuối

Bản notebook 09 này dùng cơ chế ranking như sau:

- `target_1h`, `target_24h`: xếp theo `test_daylight_RMSE`, vì đây là bài toán dự báo điểm theo giờ và cần tránh việc `test_RMSE` toàn bộ giờ bị thấp giả do ban đêm gần 0.
- `daily_total`: xếp theo `test_RMSE` hoặc `test_daily_total_RMSE`, vì đây là tổng năng lượng theo ngày.
- `multihorizon`: xếp theo `test_daylight_RMSE` của toàn chuỗi 24h.
- Các metric `test_h1_RMSE`, `test_h12_RMSE`, `test_h24_RMSE` từ notebook 07 chỉ được đưa vào bảng tham khảo, không trộn vào ranking của `target_1h`/`target_24h` trực tiếp.

Các file xuất ra nằm trong thư mục `outputs_09`:

- `09_corrected_final_results_for_report.csv`
- `09_corrected_best_by_task_dataset.csv`
- `09_corrected_source_coverage.csv`
- `09_reference_only_derived_point_from_notebook07.csv`
- `09_sorting_policy.csv`
- `09_corrected_results_synthesis.xlsx` nếu môi trường có `openpyxl`
